# RoBERTa Final Model Training
This notebook trains a RoBERTa model for sequence classification using a predefined set of optimal hyperparameters. The dataset is split into training (80%), validation (10%), and test (10%) sets.

In [19]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report
)

In [20]:
import wandb
import huggingface_hub

os.environ["WANDB_PROJECT"] = "roberta_degendered_final"

# wandb.login(key="YOUR_WANDB_KEY")
# huggingface_hub.login(token="YOUR_HF_TOKEN")

wandb.init()

In [21]:
# Initialize W&B
wandb.init()

In [22]:
# Model and Hyperparameter Configuration
model_name = "roberta-base"
model_cache_path = "../scratch/cache/roberta_degendered_final"

hyperparameters = {
    "learning_rate": 2.0e-05,
    "num_train_epochs": 6,
    "per_device_train_batch_size": 16,
    "weight_decay": 0.0
}

In [23]:
# Data Preparation (80:10:10 Split)
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# First split: 80% train, 20% temp (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Second split: 10% validation, 10% test from the temp set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)

In [24]:
# Tokenization Function
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding=False, max_length=512)
    tokens["labels"] = example["label"]
    return tokens

# Create Hugging Face Datasets
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_dataset = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_val = val_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

In [25]:
# Metrics Computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    
    # Get classification report
    report = classification_report(labels, preds, output_dict=True, zero_division=0, target_names=['Female', 'Male'])
    
    # Flatten the report for easy logging
    metrics = {
        'accuracy': report['accuracy'],
        'macro_avg_precision': report['macro avg']['precision'],
        'macro_avg_recall': report['macro avg']['recall'],
        'macro_avg_f1': report['macro avg']['f1-score'],
        'weighted_avg_precision': report['weighted avg']['precision'],
        'weighted_avg_recall': report['weighted avg']['recall'],
        'weighted_avg_f1': report['weighted avg']['f1-score'],
        'female_precision': report['Female']['precision'],
        'female_recall': report['Female']['recall'],
        'female_f1': report['Female']['f1-score'],
        'female_support': report['Female']['support'],
        'male_precision': report['Male']['precision'],
        'male_recall': report['Male']['recall'],
        'male_f1': report['Male']['f1-score'],
        'male_support': report['Male']['support']
    }
    
    print("Confusion Matrix:\n", confusion_matrix(labels, preds))

    return metrics

In [26]:
# Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Female", 1: "Male"},
    label2id={"Female": 0, "Male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
# Training Arguments
final_model_output_dir = "../scratch/final_roberta_degendered_model"
training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=hyperparameters["per_device_train_batch_size"],
    num_train_epochs=hyperparameters["num_train_epochs"],
    learning_rate=hyperparameters["learning_rate"],
    weight_decay=hyperparameters["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_avg_f1",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_roberta_degendered_training"
)

In [28]:
# Trainer Initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val, # Use validation set for in-training evaluation
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_605565/4000249224.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [29]:
# Train the Model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Avg Precision,Macro Avg Recall,Macro Avg F1,Weighted Avg Precision,Weighted Avg Recall,Weighted Avg F1,Female Precision,Female Recall,Female F1,Female Support,Male Precision,Male Recall,Male F1,Male Support
1,0.648600,0.620329,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
2,0.639200,0.617729,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
3,0.627700,0.616182,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
4,0.613900,0.622763,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
5,0.583900,0.634681,0.617353,0.575353,0.582957,0.575920,0.641133,0.617353,0.626494,0.402941,0.492806,0.443366,278.000000,0.747764,0.673108,0.708475,621.000000
6,0.488500,0.648284,0.674082,0.602953,0.588257,0.591608,0.655421,0.674082,0.661629,0.465438,0.363309,0.408081,278.000000,0.740469,0.813205,0.775134,621.000000


Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[137 141]
 [203 418]]
Confusion Matrix:
 [[101 177]
 [116 505]]


TrainOutput(global_step=2700, training_loss=0.596646412037037, metrics={'train_runtime': 176.6407, 'train_samples_per_second': 244.191, 'train_steps_per_second': 15.285, 'total_flos': 1.134790581769248e+16, 'train_loss': 0.596646412037037, 'epoch': 6.0})

In [30]:
# Final Evaluation on the Test Set
print("--- Final Evaluation on Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")

print("\nFinal Test Set Evaluation Results:")
print(test_results)

--- Final Evaluation on Test Set ---


Confusion Matrix:
 [[ 93 186]
 [149 471]]

Final Test Set Evaluation Results:
{'test_loss': 0.6739640235900879, 'test_accuracy': 0.6273637374860956, 'test_macro_avg_precision': 0.5505962489150534, 'test_macro_avg_recall': 0.546505376344086, 'test_macro_avg_f1': 0.5473360818978021, 'test_weighted_avg_precision': 0.6136750768734279, 'test_weighted_avg_recall': 0.6273637374860956, 'test_weighted_avg_f1': 0.6195303426269241, 'test_female_precision': 0.384297520661157, 'test_female_recall': 0.3333333333333333, 'test_female_f1': 0.3570057581573896, 'test_female_support': 279.0, 'test_male_precision': 0.7168949771689498, 'test_male_recall': 0.7596774193548387, 'test_male_f1': 0.7376664056382146, 'test_male_support': 620.0, 'test_runtime': 0.8167, 'test_samples_per_second': 1100.739, 'test_steps_per_second': 69.791, 'epoch': 6.0}


In [31]:
# Save the Final Model and Tokenizer
trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)
print(f"Final model and tokenizer saved to: {final_model_output_dir}")

Final model and tokenizer saved to: ../scratch/final_roberta_degendered_model
